# 1. Downloading data using cdsapi

`cdsapi` library is a Python Python interface for accessing the **Copernicus Climate Data Store (CDS)**. CDS is a comprehensive platform offering free environmental and climate datasets, including satellite observations and reanalysis products like ERA5.

To use this library, you must first:
1. Install it via `pip install cdsapi`
2. Register for a free account at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu/) to obtain an API key
3. Configure your credentials by creating a `.cdsapirc` file in your home directory with your API key

Without these credentials, the script will fail when attempting to download data, as CDS requires authentication for all requests.

## Advantages of `cdsapi`

Using `cdsapi` to fetch data directly from CDS offers several significant benefits:

**Automation and Efficiency**
- Eliminates manual downloads of large files
- Enables programmatic filtering of variables, time ranges, and regions
- Seamlessly integrates with Python workflows (Jupyter Notebooks, scripts)

**Access to High-Quality Data**
- Provides standardized, validated datasets (ERA5, CAMS, etc.)
- Ensures consistent formatting across all data products
- Eliminates need for specialized software for data acquisition

**Simplified Data Management**
- Handles authentication and request formatting automatically
- Complies with CDS usage policies and rate limits
- Scales from single files to thousands of files efficiently

**Research and Operational Benefits**
- Ideal for time-series analysis requiring multiple files
- Enables reproducible data acquisition workflows
- Reduces manual errors in data collection processes

In [1]:
from pathlib import Path

import cdsapi

# Downloads go to <repo root>/data/derived/module-00/{ERA5,CERRA,CMIP6}
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DOWNLOAD_DIR = root / "data" / "derived" / "module-00"

### Datasets and Download Scripts for Module 0

This section contains download scripts using **CDS API (cdsapi)** to obtain the core datasets required for Module 0. These scripts automate the acquisition of multiple climate and atmospheric variables that will form the basis for all geospatial analysis exercises in this module.

### Dataset Overview

For Module 0, we will work with three complementary datasets through CDS API:

1. **ERA5 Reanalysis** - Global atmospheric reanalysis providing hourly data on multiple variables at 31km resolution
2. **CERRA Reanalysis** - Climate ERA Regional Reanalysis offering high-resolution (11km) data specifically for Europe and surrounding regions
3. **CMIP6 Climate Projections** - Coupled Model Intercomparison Project Phase 6 data containing climate model simulations

These datasets have been carefully selected to provide:
- Global coverage (ERA5) and regional detail (CERRA)
- Current climate observations (ERA5/CERRA) and future projections (CMIP6)
- A comprehensive range of atmospheric variables for analysis

### Download Script Characteristics

The scripts implement a **customized download workflow** that extends standard CDS API functionality with some modifications:

1. **Targeted Variable Selection**
   Rather than downloading entire datasets, the code specifies exact variables needed:
   - ERA5: 2m temperature, precipitation, sea surface temperature
   - CERRA: 10m wind parameters, 2m humidity, orography
   - CMIP6: Near-surface temperature, humidity, wind, precipitation

   This focused approach ensures we obtain only the variables required for Module 0 exercises.

2. **Custom File Naming System**
   The most significant modification is the **custom filename generation**:
   ```python
   filename = f'{dataset_type}-AIFTraining-M0-{variable}.{extension}'
   ```
   This creates descriptive filenames like:
    - `ERA5-AIFTraining-M0-2m_temperature.grib`
    - `CERRA-AIFTraining-M0-10m_wind_speed.grib`
    - `CMIP6-AIFTraining-M0-near_surface_air_temperature.zip`
   
   These custom names replace the default generic filenames that CDS would otherwise generate, providing:

    - Immediate identification of file contents
    - Clear organization by dataset type and variable
    - Prevention of file overwrites during batch downloads
    - Self-documenting archive structure

The other features specified in the script are automatically generated in the CDS webpage when requesting the data.

These download scripts represent the first step in creating a comprehensive dataset collection that will support all exercises in Module 0, providing the necessary atmospheric and climate data in a well-organized, accessible format.

### ERA5

In [2]:
dataset = "reanalysis-era5-single-levels"

variables = ['2m_temperature', 'total_precipitation', 'sea_surface_temperature']

(DOWNLOAD_DIR / "ERA5").mkdir(parents=True, exist_ok=True)

for var in variables:

    filename = DOWNLOAD_DIR / "ERA5" / f'ERA5-AIFTraining-M0-{var}.grib'

    request = {
        "product_type": ["reanalysis"],
        "variable": var,
        "year": [
            "2020",
        ],
        "month": [
            "01",
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "01:00", "02:00",
            "03:00", "04:00", "05:00",
            "06:00", "07:00", "08:00",
            "09:00", "10:00", "11:00",
            "12:00", "13:00", "14:00",
            "15:00", "16:00", "17:00",
            "18:00", "19:00", "20:00",
            "21:00", "22:00", "23:00"
        ],
        "data_format": "grib",
        "download_format": "unarchived"
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request, str(filename))

2026-10-02 22:56:29,747 INFO Request ID is 5c135ea7-3718-4cd7-8da2-ca54a1ecc399
2026-10-02 22:56:29,810 INFO status has been updated to successful


14047c0e884a8f46df44adcfbbc28ef0.grib:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

2026-10-02 22:57:46,248 INFO Request ID is ca726e50-32cc-4578-8351-bd47797b1ca1
2026-10-02 22:57:46,325 INFO status has been updated to successful


5d020c4b7969fe8f0d71a52ac7a89ec5.grib:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

2026-10-02 22:59:18,325 INFO Request ID is 60a9e9d5-c4d2-4e56-acd7-f6415c9dc6e3
2026-10-02 22:59:18,396 INFO status has been updated to successful


a7f42bf53be7d557272b3c6218cc3419.grib:   0%|          | 0.00/1.04G [00:00<?, ?B/s]

## CERRA

In [3]:
dataset = "reanalysis-cerra-single-levels"

variables = [
        "10m_wind_direction",
        "10m_wind_speed",
        "2m_relative_humidity",
        "orography"
    ]

(DOWNLOAD_DIR / "CERRA").mkdir(parents=True, exist_ok=True)

for var in variables:

    filename = DOWNLOAD_DIR / "CERRA" / f'CERRA-AIFTraining-M0-{var}.grib'

    request = {
        "variable": var,
        "level_type": "surface_or_atmosphere",
        "data_type": ["reanalysis"],
        "product_type": "analysis",
        "year": [
            "2020",
        ],
        "month": [
            "01", 
        ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "03:00", "06:00",
            "09:00", "12:00", "15:00",
            "18:00", "21:00"
        ],
        "data_format": "grib"
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request, str(filename))


2026-10-02 23:00:10,652 INFO Request ID is 20dec6b8-c5fe-46b4-99fa-2a4497c551cf
2026-10-02 23:00:12,497 INFO status has been updated to successful


eeb252c37e7b2eb1b75cad6253e746ec.grib:   0%|          | 0.00/541M [00:00<?, ?B/s]

2026-10-02 23:00:42,570 INFO Request ID is c9fce22e-d317-4ec1-b920-ffa987b8b1d0
2026-10-02 23:00:42,720 INFO status has been updated to successful


b5c6312ca9718e51991e334c81bef187.grib:   0%|          | 0.00/541M [00:00<?, ?B/s]

2026-10-02 23:01:13,324 INFO Request ID is ba61cca5-1919-43d8-b7ad-c6ac38fa8aac
2026-10-02 23:01:13,413 INFO status has been updated to successful


8c62ec2d9042c13bec211ab4602d4e11.grib:   0%|          | 0.00/541M [00:00<?, ?B/s]

2026-10-02 23:01:39,228 INFO Request ID is e31572bd-0746-462b-98fb-51a5410e8ca9
2026-10-02 23:01:39,299 INFO status has been updated to successful


dbd0eed36a9bac55b3fbfe7044eda689.grib:   0%|          | 0.00/541M [00:00<?, ?B/s]

## CMIP6

In [4]:
dataset = "projections-cmip6"

variables = [
        "near_surface_air_temperature",
        "near_surface_specific_humidity",
        "near_surface_wind_speed",
        "precipitation",
    ]

(DOWNLOAD_DIR / "CMIP6").mkdir(parents=True, exist_ok=True)

for var in variables:

    filename = DOWNLOAD_DIR / "CMIP6" / f'CMIP6-AIFTraining-M0-{var}.zip'

    request = {
        "temporal_resolution": "daily",
        "experiment": "historical",
        "variable": var,
        "model": "ec_earth3_cc",
        "year": ["2000"],
        "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
        ]
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request, str(filename))


2026-10-02 23:02:04,525 INFO Request ID is ef0d8598-0c40-4cab-a2fa-539c16ec24ed
2026-10-02 23:02:04,595 INFO status has been updated to successful


123d5fb768a707503457741b0c161758.zip:   0%|          | 0.00/136M [00:00<?, ?B/s]

2026-10-02 23:02:13,845 INFO Request ID is 6c64f6ab-f912-4252-9030-cb14b52d017d
2026-10-02 23:02:13,926 INFO status has been updated to accepted
2026-10-02 23:02:35,265 INFO status has been updated to running
2026-10-02 23:03:03,948 INFO status has been updated to successful


4543c1165b06945dd0e54d70ce2b17d2.zip:   0%|          | 0.00/160M [00:00<?, ?B/s]

2026-10-02 23:03:16,206 INFO Request ID is bc75b289-b338-4ee4-96b6-c14cd78b73cb
2026-10-02 23:03:16,512 INFO status has been updated to accepted
2026-10-02 23:03:37,768 INFO status has been updated to running
2026-10-02 23:04:06,422 INFO status has been updated to successful


a4257e27b8dd10dabb4050b326473fa2.zip:   0%|          | 0.00/163M [00:00<?, ?B/s]

2026-10-02 23:04:15,937 INFO Request ID is 8eb83877-4d90-417a-bba0-53e76705e5ff
2026-10-02 23:04:16,018 INFO status has been updated to accepted
2026-10-02 23:04:37,410 INFO status has been updated to running
2026-10-02 23:04:48,879 INFO status has been updated to successful


2deb1ead1df2e976c3a4dbacc5b13156.zip:   0%|          | 0.00/136M [00:00<?, ?B/s]